# WS01.2 Part 1 — AEDT Mechanical (Thermal) 모델 빌드 continuation

Ansys 교육자료 **AEDT_Mech_Thermal_2025R1_EN_WS01.2** 의 워크플로우를 이어서 자동화합니다.

**전제 (이미 수동으로 완료된 부분, PDF p.3~6):**
- `Electric_Motor_Mechanical_AEDT_3D_part1.aedtz` 를 열어 Maxwell → Mechanical **Create Target Design** 완료
- Sheet object 삭제 완료
- `rotating_band_out` 를 Z축 기준 45° 섹터로 8개 duplicate 후 **Unite** 까지 완료

**이 노트북이 진행하는 단계 (PDF p.7~18):**

| Step | PDF | 내용 |
|------|-----|------|
| A | p.7 | Shaft/Rotor/Magnets/Stator/whole_Region 를 45°×8 duplicate + unite |
| A2 | p.7 | Winding 세그먼트 병합 (같은 coil 끼리, 최종 24 bodies) |
| B | p.8 | Shaft → `Steel_1008`(52 W/m-C), SweepAlongVector 를 +50mm(Z) 연장 |
| C | p.9 | `Rotating_Band_out`, `Whole_Region` → `Air` |
| D | p.10~12 | Insulation 실린더 생성 → subtract → 이름/재질(insulation, k=10, ρ=2000) |
| E | p.13 | Magnet 재질 → `N30UH_20C` |
| F | p.14~15 | 자동생성 Convection 삭제 후 Convection 재지정 (stator 1500/65, rotor·shaft top 96/110) |
| G | p.16 | `Rotating_Band_out` 에 Rotating Fluid (gap 1mm, rot_speed=1000rpm) |
| H | p.17 | Rotor–Shaft Contact (Thermal Impedance 0.005 K·m²/W) |
| I | p.18 | EM Loss thermal source (Maxwell transient, 1.875~3.75ms) |

> ⚠️ **주의:** object/face 이름은 모델에 따라 다릅니다. 각 Step 전에 **Cell 2 의 이름 목록 출력**으로 실제 body 이름을 확인하고, 설정 딕셔너리(`CFG`)를 맞춰 주세요.
> Step G(Rotating Fluid), H(Contact), I(EM Loss) 는 pyaedt 고수준 API 가 없거나 버전차가 커서 **native COM API** 로 작성되어 있습니다 — 값은 PDF 기준으로 채워 두었으나, 최초 1회는 GUI 결과와 대조해 검증하시길 권장합니다.

## 0. Config & Connect

In [ ]:
import os, sys, traceback
from ansys.aedt.core import Desktop, Mechanical

# ── 사용자 설정 ────────────────────────────────────────────────────────────
AEDT_VERSION = "2025.1"      # 워크샵 기준 2025 R1. 실행 중인 버전에 맞게 수정
NON_GRAPHICAL = False        # GUI 표시 (기존 세션에 붙을 때는 무시됨)

# 붙을 프로젝트/디자인. None 이면 활성 프로젝트 + '3번째 디자인' 자동 선택
PROJECT_NAME = None          # 예: "Electric_Motor_Mechanical_AEDT_3D_part1"
DESIGN_INDEX = 2             # 0-based. '3번째 디자인' = index 2 (Mechanical thermal)
DESIGN_NAME  = None          # 이름을 직접 지정하려면 여기에 (지정 시 DESIGN_INDEX 무시)
# ──────────────────────────────────────────────────────────────────────────

# 이미 떠 있는 AEDT 세션에 attach (새 세션을 만들지 않음)
desktop = Desktop(specified_version=AEDT_VERSION,
                  new_desktop_session=False,
                  non_graphical=NON_GRAPHICAL,
                  close_on_exit=False)

proj_list = desktop.project_list()
print(f"열린 프로젝트: {proj_list}")

proj = PROJECT_NAME or (desktop.active_project().GetName() if desktop.active_project() else proj_list[0])
oProject = desktop.odesktop.SetActiveProject(proj)
design_list = list(oProject.GetTopDesignList())
print(f"프로젝트 '{proj}' 의 디자인 목록:")
for i, dn in enumerate(design_list):
    print(f"  [{i}] {dn}")

design = DESIGN_NAME or design_list[DESIGN_INDEX]
print(f"\n선택된 디자인 → {design}")

In [ ]:
# Mechanical (Thermal) 디자인에 pyaedt 로 attach
mech = Mechanical(specified_version=AEDT_VERSION,
                  new_desktop_session=False,
                  close_on_exit=False,
                  project=proj,
                  design=design)

mdl      = mech.modeler          # 지오메트리 조작
oEditor  = mech.oeditor          # native 3D modeler
oDesign  = mech.odesign          # native design
oBnd     = oDesign.GetModule("BoundarySetup")   # 경계조건

print(f"Design type : {mech.design_type}")
print(f"Solution    : {mech.solution_type}")
print(f"Body count  : {len(mdl.object_names)}")

## 1. Body 이름 확인 (진행 전 필수)

실제 모델의 body 이름을 확인하고, 아래 Step 들의 `CFG` 를 맞춰 주세요.

In [ ]:
names = sorted(mdl.object_names)
print(f"총 {len(names)} bodies\n")
for n in names:
    print("  ", n)

# 키워드로 그룹 미리보기 (모델별 접두어 확인용)
def find(*keys):
    keys = [k.lower() for k in keys]
    return [n for n in names if any(k in n.lower() for k in keys)]

print("\n[keyword preview]")
for kw in ["shaft", "rotor", "magnet", "stator", "region", "band", "ph1", "ph2", "ph3", "cylinder"]:
    hits = find(kw)
    if hits:
        print(f"  {kw:8s}: {hits}")

## Step A (p.7) — 주요 구조물 45°×8 duplicate + unite

`rotating_band_out` 는 이미 완료. 나머지 Shaft/Rotor/Magnets/Stator/whole_Region 를 처리합니다.

> duplicate_around_axis 는 pyaedt 버전에 따라 인자명이 다르므로(`clones`/`nclones`, `axis`/`cs_axis`) wrapper 로 흡수합니다.

In [ ]:
def dup_around_z(name, angle=45, clones=8):
    """Z축 기준 angle° 로 clones 개(원본 포함) 복제. 새로 생긴 이름 리스트 반환."""
    try:
        res = mdl.duplicate_around_axis(name, axis="Z", angle=angle, clones=clones)
    except TypeError:
        res = mdl.duplicate_around_axis(name, cs_axis="Z", angle=angle, nclones=clones)
    # 반환형 정규화: (bool, [names]) 또는 [names]
    if isinstance(res, (list, tuple)) and len(res) == 2 and isinstance(res[0], (bool, int)):
        new_names = list(res[1])
    else:
        new_names = list(res)
    return new_names

def dup_and_unite(base, angle=45, clones=8):
    """base body 를 복제 후 전체를 하나로 unite. 결과 body 이름 반환."""
    if base not in mdl.object_names:
        print(f"  [SKIP] '{base}' 없음")
        return None
    new_names = dup_around_z(base, angle, clones)
    group = [base] + new_names
    mdl.unite(group)
    print(f"  [OK] {base}: {len(group)} segments → unite")
    return base

# ── 처리할 주요 body 접두어 (모델 이름에 맞게 수정) ─────────────────────────
MAIN_BODIES = ["Shaft_1", "Rotor_Lamination_1", "Stator_Lamination_1"]
# 자석은 여러 극이 있으므로 키워드로 수집하여 각각 처리
MAGNET_BODIES = find("magnet")
REGION_BODIES = find("whole_region", "region")
# ──────────────────────────────────────────────────────────────────────────

print("[Main structural bodies]")
for b in MAIN_BODIES:
    dup_and_unite(b)

print("[Magnets]")
for b in list(MAGNET_BODIES):
    dup_and_unite(b)

print("[Region]")
for b in list(REGION_BODIES):
    dup_and_unite(b)

## Step A2 (p.7) — Winding 병합 (최종 24 bodies)

PDF 기준:
- `Ph1_P2_C1*` (8개) : 이미 완전한 coil → 그대로 유지
- `Ph2_P2_C1*` + `Ph2_P2_C2*` : 대응끼리 병합 → 8개
- `Ph3_P1_C1*` + `Ph3_P1_C2*` : 대응끼리 병합 → 8개

실제 접미어(위치 인덱스)는 모델마다 다르므로, **먼저 winding 이름을 출력**해 규칙을 확인한 뒤 병합 로직을 맞추세요.

In [ ]:
windings = [n for n in mdl.object_names if n.lower().startswith(("ph1", "ph2", "ph3"))]
windings.sort()
print(f"Winding bodies: {len(windings)}")
for w in windings:
    print("  ", w)

In [ ]:
import re
from collections import defaultdict

def coil_key(name):
    """같은 coil 로 묶을 key. 'C1'/'C2' 세그먼트 구분자를 제거해 병합쌍을 매칭.
    필요하면 이 규칙을 모델 naming 에 맞게 수정하세요."""
    # 예: Ph2_P2_C1_3 / Ph2_P2_C2_3  →  같은 (Ph2, P2, 3) 로 묶기
    m = re.match(r"(Ph\d+)_P(\d+)_C\d+(.*)", name)
    if m:
        return (m.group(1), m.group(2), m.group(3))
    return (name,)

groups = defaultdict(list)
for w in windings:
    groups[coil_key(w)].append(w)

print("[병합 그룹 미리보기]")
for k, v in sorted(groups.items()):
    tag = "UNITE" if len(v) > 1 else "keep "
    print(f"  {tag} {k}: {v}")

# 실제 병합 실행 (미리보기 확인 후 주석 해제)
# for k, v in groups.items():
#     if len(v) > 1:
#         mdl.unite(v)
#         print(f"  united {k} -> {v[0]}")
# print(f"최종 winding bodies: {len([n for n in mdl.object_names if n.lower().startswith(('ph1','ph2','ph3'))])}")

## Step B (p.8) — Shaft 재질 + SweepAlongVector +50mm

- Shaft → **Steel_1008** (Project library 의 k=52 W/m-C 짜리)
- Shaft 히스토리의 `SweepAlongVector` 를 Z 방향 +50mm 연장

In [ ]:
SHAFT = "Shaft_1"   # 실제 이름 확인

# 재질 지정
try:
    mech.assign_material([SHAFT], "Steel_1008")
    print(f"[OK] {SHAFT} → Steel_1008")
except Exception:
    mdl[SHAFT].material_name = "Steel_1008"
    print(f"[OK] {SHAFT}.material_name = Steel_1008 (fallback)")

# SweepAlongVector 히스토리 편집: Z 벡터를 +50mm 연장
# pyaedt history 트리로 접근 시도 → 실패하면 GUI 에서 수동 편집 안내
try:
    hist = mdl[SHAFT].history()
    sweep_node = None
    for cname, cnode in hist.children.items():
        if "SweepAlongVector" in cname:
            sweep_node = cnode
            break
    if sweep_node is not None:
        print(f"찾은 operation: {cname}")
        print(f"현재 props keys: {list(sweep_node.props.keys())}")
        # 벡터 관련 prop 이름은 버전마다 다름 (예: 'Vector Z', 'ZDir', ...)
        # 확인 후 아래처럼 수정: sweep_node.props['<Zprop>'] = '<기존값>+50mm'
    else:
        print("[MANUAL] SweepAlongVector operation 을 히스토리에서 못 찾음 → GUI 에서 직접 +50mm")
except Exception as e:
    print(f"[MANUAL] 히스토리 자동편집 실패({e}) → GUI: Shaft 히스토리 > SweepAlongVector 벡터 Z 에 +50mm")

## Step C (p.9) — Rotating_Band_out, Whole_Region → Air

In [ ]:
AIR_BODIES = ["Rotating_Band_out", "Whole_Region"]   # 실제 이름 확인
for b in AIR_BODIES:
    if b in mdl.object_names:
        try:
            mech.assign_material([b], "Air")
        except Exception:
            mdl[b].material_name = "Air"
        print(f"[OK] {b} → Air")
    else:
        print(f"[SKIP] {b} 없음")

## Step D (p.10~12) — Insulation 실린더 생성/subtract/재질

1. Cylinder1: 원점 중심, r=95mm, h=75mm(+Z)
2. Cylinder2: r=66mm (Cylinder1 복사 후 반경 변경)
3. Cylinder1 −= Cylinder2
4. Cylinder1 −= Stator_Lamination (clone tool: stator 유지)
5. Cylinder1 −= Windings (clone tool: winding 유지)
6. Cylinder1 → 이름 `Insulation`, 재질 `insulation`(k=10, ρ=2000)

In [ ]:
# create_cylinder 인자 순서: (orientation, origin, radius, height, ...) — 버전차 wrapper
def make_cyl(name, radius, height=75):
    try:
        return mdl.create_cylinder("Z", [0, 0, 0], radius, height, name=name, material="vacuum")
    except TypeError:
        return mdl.create_cylinder(cs_axis="Z", position=[0, 0, 0], radius=radius,
                                   height=height, name=name, matname="vacuum")

cyl1 = make_cyl("Cylinder1", 95, 75)
cyl2 = make_cyl("Cylinder2", 66, 75)
print(f"생성: {cyl1.name}, {cyl2.name}")

# Cylinder1 − Cylinder2  (tool 유지 불필요)
mdl.subtract("Cylinder1", "Cylinder2", keep_originals=False)

STATOR = "Stator_Lamination_1"   # 실제 이름 확인
WINDINGS = [n for n in mdl.object_names if n.lower().startswith(("ph1", "ph2", "ph3"))]

# Cylinder1 − Stator (clone: stator 유지)
mdl.subtract("Cylinder1", STATOR, keep_originals=True)
# Cylinder1 − Windings (clone: winding 유지)
mdl.subtract("Cylinder1", WINDINGS, keep_originals=True)
print("[OK] subtract 완료")

In [ ]:
# 이름 변경 + insulation 재질 생성/지정
mdl["Cylinder1"].name = "Insulation"

if not mech.materials.checkifmaterialexists("insulation"):
    m = mech.materials.add_material("insulation")
    m.thermal_conductivity = 10      # W/m-C
    m.mass_density = 2000            # kg/m^3
    print("[OK] 재질 'insulation' 생성 (k=10, rho=2000)")
else:
    print("재질 'insulation' 이미 존재")

try:
    mech.assign_material(["Insulation"], "insulation")
except Exception:
    mdl["Insulation"].material_name = "insulation"
print("[OK] Insulation body → insulation 재질")

## Step E (p.13) — Magnet 재질 → N30UH_20C

In [ ]:
MAGNETS = find("magnet")   # Step A 에서 unite 된 자석 body
for b in MAGNETS:
    try:
        mech.assign_material([b], "N30UH_20C")
    except Exception:
        mdl[b].material_name = "N30UH_20C"
    print(f"[OK] {b} → N30UH_20C")

## Step F (p.14~15) — Convection 경계조건

- 자동 생성된 Convection 삭제
- Stator lamination 외경 표면 → h=1500 W/m²·C, T=65°C
- Rotor top face, Shaft top face → h=96 W/m²·C, T=110°C

face 선택은 body 표면 id 로 하며, 아래 helper 로 외경/상단면을 추정합니다. **선택된 face 를 GUI 에서 확인**하세요.

In [ ]:
# 자동 생성된 convection 삭제
try:
    for b in list(mech.boundaries):
        if "conv" in b.name.lower():
            b.delete()
            print(f"삭제: {b.name}")
except Exception as e:
    print(f"자동 convection 삭제 스킵: {e}")

def outer_lateral_face(body):
    """반경이 가장 큰 측면(외경) face id 추정."""
    fid = mdl.get_faceid_from_position  # placeholder guard
    faces = mdl[body].faces
    # 중심축(Z)에서 가장 먼 face 중심을 외경으로 간주
    best, best_r = None, -1
    for f in faces:
        cx, cy, cz = f.center
        r = (cx**2 + cy**2) ** 0.5
        if r > best_r:
            best_r, best = r, f.id
    return best

def top_face(body):
    """Z 가 가장 큰 상단 face id."""
    best, best_z = None, -1e18
    for f in mdl[body].faces:
        cz = f.center[2]
        if cz > best_z:
            best_z, best = cz, f.id
    return best

In [ ]:
# Stator 외경 convection: 1500 W/m2C, 65C
stator_face = outer_lateral_face(STATOR)
mech.assign_uniform_convection([stator_face], convection_value=1500,
                               convection_units="w_per_m2kel",
                               temperature="65cel",
                               boundary_name="Conv_Stator")
print(f"[OK] Stator conv (face {stator_face}) 1500/65")

# Rotor top / Shaft top convection: 96 W/m2C, 110C
ROTOR = "Rotor_Lamination_1"   # 실제 이름 확인
for body, bname in [(ROTOR, "Conv_RotorTop"), (SHAFT, "Conv_ShaftTop")]:
    f = top_face(body)
    mech.assign_uniform_convection([f], convection_value=96,
                                   convection_units="w_per_m2kel",
                                   temperature="110cel",
                                   boundary_name=bname)
    print(f"[OK] {bname} (face {f}) 96/110")

## Step G (p.16) — Rotating Fluid (native API)

`Rotating_Band_out` 에 Rotating Fluid 경계: Gap=1mm, Axis=Z, Rotor Position=Inner, Speed=`rot_speed`(초기 1000rpm).

> pyaedt 고수준 메서드가 없어 **native `oBnd`** 로 지정합니다. arg 스키마는 AEDT 버전에 따라 다를 수 있으니, 최초 1회는 GUI 에서 한 번 지정→기록(Record Script)한 뒤 대조 권장.

In [ ]:
# rot_speed 프로젝트 변수 (초기 1000 rpm)
try:
    mech["rot_speed"] = "1000rpm"
    print("변수 rot_speed = 1000rpm")
except Exception as e:
    print(f"변수 설정 스킵: {e}")

ROT_BAND = "Rotating_Band_out"
try:
    oBnd.AssignRotatingFluid(
        [
            "NAME:RotatingFluid1",
            "Objects:=", [ROT_BAND],
            "Gap Thickness:=", "1mm",
            "Rotational Speed:=", "rot_speed",
            "Axis Direction:=", "Z",
            "Rotor Position:=", "Inner",
        ]
    )
    print("[OK] Rotating Fluid 지정")
except Exception as e:
    print(f"[MANUAL] Rotating Fluid 자동지정 실패 → GUI 로 지정하세요.\n  {e}")
    traceback.print_exc()

## Step H (p.17) — Rotor–Shaft Contact (native API)

Rotor 와 Shaft 접촉면에 Thermal Impedance = **0.005 K·m²/W**.

In [ ]:
# Rotor–Shaft 접촉면 (Rotor 안쪽 원통면) 추정 — GUI 확인 필요
def inner_lateral_face(body):
    best, best_r = None, 1e18
    for f in mdl[body].faces:
        cx, cy, cz = f.center
        r = (cx**2 + cy**2) ** 0.5
        if 0 < r < best_r:
            best_r, best = r, f.id
    return best

contact_face = inner_lateral_face(ROTOR)
try:
    oContact = oDesign.GetModule("Contact")
    oContact.AssignContact(
        [
            "NAME:Contact1",
            "Faces:=", [contact_face],
            "Resistance Type:=", "Thermal Impedance",
            "Thermal Impedance:=", "0.005",
        ]
    )
    print(f"[OK] Contact (face {contact_face}) Thermal Impedance 0.005")
except Exception as e:
    print(f"[MANUAL] Contact 자동지정 실패 → GUI 로 지정.\n  {e}")
    traceback.print_exc()

## Step I (p.18) — EM Loss thermal source (Maxwell transient)

Magnet/Winding/Rotor/Stator body 에 Maxwell 3D transient 결과의 손실을 import.
- Use This Project, Map Variable by name
- Intrinsics(Time): Start 1.875ms, End 3.75ms (3 torque period 평균)

> pyaedt `assign_em_losses` 는 transient 시간평균 스키마와 차이가 있어, 먼저 pyaedt 시도 후 실패 시 native `AssignEMLoss` 를 사용합니다. **Maxwell 디자인/솔루션 이름**을 실제 값으로 넣으세요.

In [ ]:
# ── EM Loss 소스 설정값 ────────────────────────────────────────────────────
MAXWELL_DESIGN = design_list[0]     # 보통 Maxwell 디자인이 첫번째. 확인 후 수정
MAXWELL_SETUP  = "Setup1"           # Maxwell transient setup 이름
EMLOSS_BODIES  = find("magnet") + WINDINGS + [ROTOR, STATOR]
START_TIME, END_TIME = "1.875ms", "3.75ms"
# ──────────────────────────────────────────────────────────────────────────
print(f"EM Loss 대상 {len(EMLOSS_BODIES)} bodies")
print(f"소스: {MAXWELL_DESIGN} / {MAXWELL_SETUP}")

# 자동 생성된 EM Loss 삭제
try:
    for b in list(mech.boundaries):
        if "emloss" in b.name.lower() or "em loss" in b.name.lower():
            b.delete()
            print(f"삭제: {b.name}")
except Exception as e:
    print(f"자동 EM Loss 삭제 스킵: {e}")

assigned = False
try:
    mech.assign_em_losses(
        designname=MAXWELL_DESIGN,
        setupname=MAXWELL_SETUP,
        sweepname="LastAdaptive",
        object_list=EMLOSS_BODIES,
    )
    assigned = True
    print("[OK] assign_em_losses (pyaedt)")
except Exception as e:
    print(f"[pyaedt 실패] {e} → native AssignEMLoss 시도")

if not assigned:
    try:
        oBnd.AssignEMLoss(
            [
                "NAME:EMLoss1",
                "Objects:=", EMLOSS_BODIES,
                "Project:=", "This Project*",
                "Product:=", "Maxwell",
                "Design:=", MAXWELL_DESIGN,
                "Soln:=", f"{MAXWELL_SETUP} : Transient",
                "Simulate source design as needed:=", True,
                "Preserve source design solution:=", True,
                "Intrinsics:=", ["Start:=", START_TIME, "Stop:=", END_TIME],
                ["NAME:Variable Mapping"],  # Map Variable by name 은 GUI 에서 확인
            ]
        )
        print("[OK] AssignEMLoss (native)")
    except Exception as e:
        print(f"[MANUAL] EM Loss 자동지정 실패 → GUI 로 지정.\n  {e}")
        traceback.print_exc()

## 저장

In [ ]:
mech.save_project()
print("프로젝트 저장 완료")
# 세션은 유지 (close_on_exit=False). 필요 시 desktop.release_desktop(False, False)